# AI-Based Market Condition Detection and Risk Monitoring System
## Explainable historical market-regime and portfolio-risk analysis

This notebook is the faculty-facing companion to `train_model.py`. It deliberately calls the same saved-output pipeline as the Streamlit dashboard, preventing inconsistent model results. It performs historical analysis only; it does not forecast prices.


## Methodology

Features are NIFTY 50 daily return, 20-day rolling volatility, 20-day momentum, India VIX, and drawdown. The earliest 80% of rows fit `StandardScaler` and K-Means. The final 20% is transformed and assigned by that fitted model only. K=3 is retained for understandable low/medium/high volatility labels; silhouette is a separation measure, not an accuracy score.


In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd()
if not (PROJECT / 'train_model.py').exists():
    raise FileNotFoundError('Open this notebook from the Market_Risk_Intelligence directory.')
print('Project directory:', PROJECT)


## Run the reproducible pipeline

The default run uses the supplied raw NIFTY/VIX data and downloads the ten NSE stock series if no cached prices are available. If network access is unavailable, use the second command to create only the valid regime outputs.


In [ ]:
# Default: creates all outputs when the stock download succeeds
!{sys.executable} train_model.py

# Offline alternative (uncomment if needed):
# !{sys.executable} train_model.py --skip-stock-download


In [ ]:
import pandas as pd
import plotly.express as px

out = PROJECT / 'outputs'
results = pd.read_csv(out / 'regime_results.csv', parse_dates=['Date'])
summary = pd.read_csv(out / 'regime_summary.csv')
evaluation = pd.read_csv(out / 'model_evaluation.csv')
display(summary)
px.scatter(results, x='Date', y='NIFTY50', color='Regime', title='Historical NIFTY 50 market regimes').show()


## Regime transitions

Rows are the current historical regime and columns are the next observed regime. A probability is the share of observed transitions out of that row; it is not a promise about a future transition.


In [ ]:
transition = pd.read_csv(out / 'transition_matrix.csv', index_col=0)
px.imshow(transition, text_auto='.1%', color_continuous_scale='Blues', title='Chronological regime transition probabilities').show()
display(summary[['Regime', 'Percentage_of_Time', 'Average_Consecutive_Days']])


## Model evaluation

PCA below is a two-dimensional visualisation of the scaled feature space; it is not used to fit K-Means.


In [ ]:
display(evaluation)
display(pd.read_csv(out / 'cluster_profiles.csv'))
px.line(evaluation, x='K', y='Inertia', markers=True, title='Elbow comparison').show()
px.bar(evaluation, x='K', y='Train_Silhouette', title='Training silhouette comparison').show()
px.scatter(results, x='PC1', y='PC2', color='Regime', symbol='Dataset_Partition', title='PCA regime visualisation').show()


## Portfolio risk

When the ten stock series are available, the output uses their equal-weighted, consecutive daily return series for wealth and maximum drawdown. Regime-specific risk numbers are conditional descriptive statistics and should not be treated as a continuous strategy.


In [ ]:
portfolio_file = out / 'portfolio_returns.csv'
if portfolio_file.exists():
    portfolio = pd.read_csv(portfolio_file, parse_dates=['Date'])
    display(pd.read_csv(out / 'portfolio_risk.csv'))
    display(pd.read_csv(out / 'regime_risk_comparison.csv'))
    px.line(portfolio, x='Date', y='Wealth', title='Equal-weighted portfolio wealth').show()
    px.line(portfolio, x='Date', y='Drawdown', title='Portfolio drawdown').show()
else:
    print('Portfolio outputs are unavailable because valid ten-stock price data was not downloaded/cached. No values have been fabricated.')
